# Agent Framework + AgentMemory Integration

## 🎯 Objective
Demonstrate how **AgentMemory** automatically integrates with **Agent Framework** through the `ContextProvider` interface.

## Key Concepts
- **ContextProvider**: Automatic memory management - no manual calls needed
- **before_run()**: Automatically loads context before agent processes query
- **after_run()**: Automatically stores response in memory
- **Multi-Session Learning**: Agent recalls and personalizes across sessions

## Demo Scenario
A financial advisor that learns about a user (Sarah) across 3 sessions:
1. **Session 1**: Initial consultation → builds profile
2. **Session 2**: Investment strategy → recalls Session 1 context
3. **Session 3**: Tax planning → uses all accumulated knowledge

## Step 1: Setup and Imports
Configure the notebook environment and import all required libraries for Agent Framework and AgentMemory integration.

In [1]:
import asyncio
import os
import sys
import time
from pathlib import Path

# Setup UTF-8 encoding for Windows
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")

# Setup paths
BASE_DIR = Path.cwd()
print(f"📁 Working directory: {BASE_DIR}")

# Find project root
while BASE_DIR != BASE_DIR.parent:
    if (BASE_DIR / "pyproject.toml").exists():
        print(f"✅ Found project root: {BASE_DIR}")
        break
    BASE_DIR = BASE_DIR.parent

sys.path.insert(0, str(BASE_DIR))

# Load environment variables
from dotenv import load_dotenv
env_path = BASE_DIR / '.env'
if env_path.exists():
    load_dotenv(env_path)
    print(f"✅ Loaded .env from: {env_path}")
else:
    print(f"⚠️  .env file not found at: {env_path}")

print("\n✅ Step 1 Complete: Environment configured")

📁 Working directory: c:\LabFiles\agent-memory\demo
✅ Found project root: c:\LabFiles\agent-memory
✅ Loaded .env from: c:\LabFiles\agent-memory\.env

✅ Step 1 Complete: Environment configured


## Step 2: Validate Azure Configuration
Verify that all required Azure OpenAI environment variables are properly configured.

In [2]:
print("=" * 70)
print("STEP 2: Validate Azure Configuration")
print("=" * 70)

# Validate required environment variables
required_vars = [
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_REASONING_MODEL",
    "AZURE_OPENAI_EMB_DEPLOYMENT"
]

print("\n🔍 Checking Azure configuration...")
missing = []
for var in required_vars:
    value = os.getenv(var)
    if not value:
        missing.append(var)
        print(f"   ❌ {var}: NOT SET")
    else:
        if "KEY" in var or "ENDPOINT" in var:
            display = value[:20] + "..." if len(value) > 20 else value
        else:
            display = value
        print(f"   ✅ {var}: {display}")

if missing:
    print(f"\n⚠️  Missing variables: {', '.join(missing)}")
else:
    print("\n✅ All Azure variables configured!")

# Demo configuration
USER_ID = "sarah_demo"
DB_PATH = str(BASE_DIR / "demo_financial_advisor.db")

print(f"\n📊 Configuration:")
print(f"   User ID: {USER_ID}")
print(f"   Database: {DB_PATH}")

# Clean up previous demo (with retry for Windows file locks)
if os.path.exists(DB_PATH):
    for attempt in range(5):
        try:
            os.remove(DB_PATH)
            print(f"   ✅ Cleaned up previous database")
            break
        except PermissionError:
            if attempt < 4:
                time.sleep(0.5)
            else:
                print(f"   ⚠️  Could not delete previous database (file in use)")

print("\n✅ Step 2 Complete: Azure validated")

STEP 2: Validate Azure Configuration

🔍 Checking Azure configuration...
   ✅ AZURE_OPENAI_ENDPOINT: https://openai-23196...
   ✅ AZURE_OPENAI_API_KEY: 7LC2kRlisRQZEqRunw2i...
   ✅ AZURE_OPENAI_REASONING_MODEL: gpt-5.1
   ✅ AZURE_OPENAI_EMB_DEPLOYMENT: text-embedding-3-large

✅ All Azure variables configured!

📊 Configuration:
   User ID: sarah_demo
   Database: c:\LabFiles\agent-memory\demo_financial_advisor.db
   ✅ Cleaned up previous database

✅ Step 2 Complete: Azure validated


## Step 3: Import Agent Framework and Memory Libraries
Import all required components: Azure credentials, Agent Framework, and AgentMemory.

In [3]:
print("=" * 70)
print("STEP 3: Import Libraries")
print("=" * 70)

# Azure and OpenAI
try:
    from azure.identity import DefaultAzureCredential
    from openai import AzureOpenAI
    print("✅ Imported: Azure authentication and OpenAI")
except ImportError as e:
    print(f"❌ Error: {e}")
    raise

# Agent Framework
try:
    from agent_framework import Agent
    from agent_framework.azure import AzureOpenAIChatClient
    print("✅ Imported: Agent Framework")
except ImportError as e:
    print(f"❌ Error: {e}")
    raise

# AgentMemory
try:
    from memory import AgentMemory, AgentMemoryConfig
    print("✅ Imported: AgentMemory")
except ImportError as e:
    print(f"❌ Error: {e}")
    raise

print("\n✅ Step 3 Complete: All imports successful")

STEP 3: Import Libraries
✅ Imported: Azure authentication and OpenAI
✅ Imported: Agent Framework
✅ Imported: AgentMemory

✅ Step 3 Complete: All imports successful


## Step 4: Define Financial Advisor Tools
Create lightweight tool functions that the agent can call. These provide financial data about retirement contribution limits.

In [4]:
print("=" * 70)
print("STEP 4: Define Financial Tools")
print("=" * 70)

def get_401k_limit(year: int = 2025) -> str:
    """Get 401k contribution limits for a given year."""
    limits = {
        2024: "$23,000 (under 50), $30,500 (50+)",
        2025: "$23,500 (under 50), $31,000 (50+)"
    }
    result = limits.get(year, "Information not available")
    return result


def get_roth_ira_limit(year: int = 2025) -> str:
    """Get Roth IRA contribution limits."""
    return "$7,000 (under 50), $8,000 (50+)"


print("\n✅ Defined tools:")
print("   • get_401k_limit(year)")
print("   • get_roth_ira_limit(year)")
print("\n✅ Step 4 Complete: Tools ready")

STEP 4: Define Financial Tools

✅ Defined tools:
   • get_401k_limit(year)
   • get_roth_ira_limit(year)

✅ Step 4 Complete: Tools ready


## Step 5: Initialize Azure OpenAI and AgentMemory
Create Azure OpenAI client and configure AgentMemory with auto-enrichment keywords for automatic context injection.

In [5]:
print("=" * 70)
print("STEP 5: Setup Azure OpenAI Client and AgentMemory")
print("=" * 70)

# Initialize Azure OpenAI client
print("\n🤖 Creating Azure OpenAI client...")
try:
    openai_client = AzureOpenAI(
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-08-01-preview")
    )
    print("✅ Azure OpenAI client initialized")
except Exception as e:
    print(f"❌ Error: {e}")
    raise

# Create AgentMemory configuration
print("\n⚙️  Creating AgentMemoryConfig...")
config = AgentMemoryConfig(
    # Auto-enrichment with keyword triggers
    auto_enrich_context=True,
    enrichment_trigger_keywords=[
        "remember", "recall", "previous", "last time", "before",
        "discussed", "mentioned", "told you", "my profile",
        "based on", "given my", "considering my"
    ],
    # Long-term synthesis - create user profile after every session
    longterm_synthesis_frequency=1,
    # Session management
    auto_manage_sessions=False,
)
print("✅ Config created with auto-enrichment enabled")

# Initialize AgentMemory
print("\n🧠 Creating AgentMemory...")
memory = AgentMemory(
    user_id=USER_ID,
    openai_client=openai_client,
    db_path=DB_PATH,
    config=config,
)
print("✅ AgentMemory initialized")
print(f"   Database: {DB_PATH}")

print("\n✅ Step 5 Complete: Memory configured")

STEP 5: Setup Azure OpenAI Client and AgentMemory

🤖 Creating Azure OpenAI client...
✅ Azure OpenAI client initialized

⚙️  Creating AgentMemoryConfig...
✅ Config created with auto-enrichment enabled

🧠 Creating AgentMemory...
✅ AgentMemory initialized
   Database: c:\LabFiles\agent-memory\demo_financial_advisor.db

✅ Step 5 Complete: Memory configured


## Step 6: Create Agent with Automatic Memory Integration
Create the Agent Framework agent with AgentMemory as a context provider. This is the key integration point where automatic memory management is enabled.

In [6]:
print("=" * 70)
print("STEP 6: Create Agent with Memory Integration")
print("=" * 70)

# Setup Azure credentials
print("\n🔐 Setting up Azure credentials...")
try:
    credential = DefaultAzureCredential()
    print("✅ Credentials configured")
except Exception as e:
    print(f"⚠️  Warning: {e}")

# Create chat client
print("\n💬 Creating Azure Chat Client...")
try:
    chat_client = AzureOpenAIChatClient(
        credential=credential,
        endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        deployment_name=os.getenv("AZURE_OPENAI_REASONING_MODEL", "gpt-4o")
    )
    print("✅ Chat client created")
except Exception as e:
    print(f"❌ Error: {e}")
    raise

# Create agent with context_providers=[memory]
print("\n🤖 Creating Agent with memory integration...")
print("   Key: context_providers=[memory]")
print("   Effect: Automatic memory management!")

agent = Agent(
    client=chat_client,
    instructions="""You are an expert financial advisor specializing in retirement planning.

Your approach:
- Provide personalized advice based on client's profile
- Reference details from previous conversations when relevant
- Explain complex concepts clearly
- Be proactive about suggesting strategies

Always be professional, accurate, and personalized.""",
    tools=[get_401k_limit, get_roth_ira_limit],
    context_providers=[memory],  # ← AUTOMATIC MEMORY INTEGRATION!
)

print("✅ Agent created with memory integration")
print("\n✅ Step 6 Complete: Agent ready")

STEP 6: Create Agent with Memory Integration

🔐 Setting up Azure credentials...
✅ Credentials configured

💬 Creating Azure Chat Client...
✅ Chat client created

🤖 Creating Agent with memory integration...
   Key: context_providers=[memory]
   Effect: Automatic memory management!
✅ Agent created with memory integration

✅ Step 6 Complete: Agent ready


## Step 7: Define Session Helper Function
Create the helper function that manages multi-turn conversations with automatic memory lifecycle (start → run queries → end with reflection).

In [7]:
print("=" * 70)
print("STEP 7: Define Session Helper Function")
print("=" * 70)

async def run_session(agent, memory, queries, session_name):
    """
    Run a conversation session with automatic memory management.
    
    Memory handling:
    - before_run() → loads context automatically
    - after_run() → stores response automatically
    - end_session() → triggers reflection and insight extraction
    """
    print(f"\n{'='*70}")
    print(f"SESSION: {session_name}")
    print(f"{'='*70}")
    
    # Start a new session
    await memory.start_session()
    session_id = memory.session_id[:8]
    print(f"✓ Session ID: {session_id}...")
    
    # Show loaded memory context
    try:
        context = await memory.get_context()
        if context and context.strip():
            context_preview = context[:300] + "..." if len(context) > 300 else context
            print(f"\n📚 Memory context loaded ({len(context)} chars):")
            for line in context_preview.split('\n')[:3]:
                if line.strip():
                    print(f"   {line[:70]}")
        else:
            print("\n📚 No previous memory (first session)")
    except Exception as e:
        print(f"⚠️  Could not load context: {type(e).__name__}")
    
    # Process queries
    print(f"\n{'─'*70}")
    for i, query in enumerate(queries, 1):
        print(f"\n  Turn {i}:")
        print(f"  👤 User: {query}")
        
        try:
            # Call agent - memory injection happens automatically
            response = await agent.run(query)
            
            # Safely extract response text
            response_text = str(response)
            if hasattr(response, 'text'):
                response_text = response.text
            elif hasattr(response, 'content'):
                response_text = response.content
            
            # Show truncated response
            display_text = response_text[:200] + "..." if len(response_text) > 200 else response_text
            print(f"  🤖 Advisor: {display_text}")
            
        except Exception as turn_err:
            print(f"  ❌ Error in turn {i}: {type(turn_err).__name__}")
            raise
    
    # End session
    print(f"\n{'─'*70}")
    try:
        result = await memory.end_session()
        insights_count = 0
        
        if isinstance(result, dict) and 'insights_extracted' in result:
            insights_count = len(result.get('insights_extracted', []))
        
        print(f"✓ Session ended ({len(queries)} turns)")
        print(f"  💡 Insights extracted: {insights_count}")
        
    except Exception as end_err:
        print(f"⚠️  Session ended with warning: {type(end_err).__name__}")

print("✅ Helper function defined")
print("\n✅ Step 7 Complete: Session helper ready")

STEP 7: Define Session Helper Function
✅ Helper function defined

✅ Step 7 Complete: Session helper ready


## Step 8: Run the Three-Session Demo
Execute the multi-session financial advisory demo showing automatic memory management across sessions.

In [12]:
print("\n" + "=" * 70)
print("STEP 8: Execute Multi-Session Demo")
print("=" * 70)

async def execute_all_sessions():
    """Execute all three sessions."""
    
    # SESSION 1: Initial Consultation
    print(f"\n{'='*70}")
    print("SESSION 1: Initial Consultation - Build User Profile")
    print(f"{'='*70}")
    
    try:
        await run_session(
            agent=agent,
            memory=memory,
            session_name="Initial Consultation",
            queries=[
                "Hi! I'm Sarah, 35 years old, software engineer making $150,000/year.",
                "I'm comfortable with moderate-to-high risk since I have 30 years until retirement.",
                "My employer offers a 401k with 4% match. What's the best strategy?"
            ]
        )
    except Exception as e:
        print(f"SESSION 1 FAILED: {e}")
        return
    
    # SESSION 2: Investment Strategy (agent recalls Session 1)
    print(f"\n{'='*70}")
    print("SESSION 2: Investment Strategy - Agent Recalls Previous Context")
    print(f"{'='*70}")
    
    try:
        await run_session(
            agent=agent,
            memory=memory,
            session_name="Investment Strategy",
            queries=[
                "Based on what we discussed before, what asset allocation do you recommend?",
                "Should I include international stocks given my risk tolerance?",
            ]
        )
    except Exception as e:
        print(f"SESSION 2 FAILED: {e}")
        return
    
    # SESSION 3: Tax Planning (agent uses accumulated context)
    print(f"\n{'='*70}")
    print("SESSION 3: Tax Planning - Agent Uses All Accumulated Knowledge")
    print(f"{'='*70}")
    
    try:
        await run_session(
            agent=agent,
            memory=memory,
            session_name="Tax Planning",
            queries=[
                "Given my income and the retirement accounts we discussed, how can I optimize taxes?",
                "Should I consider Roth conversions based on my profile?",
            ]
        )
    except Exception as e:
        print(f"SESSION 3 FAILED: {e}")
        return
    
    print(f"\n{'='*70}")
    print("✅ ALL SESSIONS COMPLETE!")
    print("   Memory was automatically managed via context_providers")
    print(f"{'='*70}")

# Run all sessions using await (not asyncio.run - Jupyter already has event loop)
try:
    await execute_all_sessions()
    print("\n✅ Step 8 Complete: All sessions executed successfully")
except Exception as e:
    print(f"\n❌ Error: {type(e).__name__}")
    print(f"   {str(e)[:200]}")


STEP 8: Execute Multi-Session Demo

SESSION 1: Initial Consultation - Build User Profile

SESSION: Initial Consultation
Loaded sqlite-vec via Python package
[Orchestrator] Starting session d985466e-0f5c-42ad-b919-21eaf72fb0be
[MemoryKeeper] Initializing session context for user: sarah_demo
  ℹ No long-term insight found for user (will be created after sufficient sessions)
  ✓ Loaded 0 recent session summaries
✓ Session ID: d985466e...

📚 Memory context loaded (51 chars):
   <session_initialization>
   </session_initialization>

──────────────────────────────────────────────────────────────────────

  Turn 1:
  👤 User: Hi! I'm Sarah, 35 years old, software engineer making $150,000/year.
[MemoryKeeper] Added user turn. Buffer: 1/6
[MemoryKeeper] Added assistant turn. Buffer: 2/6
  🤖 Advisor: Nice to meet you, Sarah. Thanks for sharing some key details—that’s a great starting point.

Before I dive into specific retirement strategies, I’d like to understand a bit more so I can tailor ever

## Step 9: Inspect Memory and Validate Results
Examine the stored memory data including session history, extracted insights, long-term profiles, and semantic search capabilities.

In [13]:
print("=" * 70)
print("STEP 9: Inspect Stored Memory")
print("=" * 70)

async def inspect_memory():
    """Inspect and display stored memory."""
    try:
        # Start inspection session
        await memory.start_session()
        
        # Demonstrate memory search
        print("\n🔍 Searching: 'What is Sarah's risk tolerance?'")
        result = await memory.search("What is Sarah's risk tolerance?")
        if result:
            print(f"   ✅ Found: {result[:150]}...")
        else:
            print("   ℹ️  No search results (embeddings may not be available)")
        
        # Show sessions
        print("\n📅 Sessions:")
        sessions = await memory.get_sessions()
        print(f"   Total sessions: {len(sessions)}")
        for i, s in enumerate(sessions, 1):
            summary = s.get('summary', '')[:80]
            print(f"   {i}. {summary}...")
        
        # Show insights
        print("\n💡 Extracted Insights:")
        insights = await memory.get_insights()
        print(f"   Total insights: {len(insights)}")
        for i, insight in enumerate(insights[:5], 1):
            insight_text = insight.get('insight_text', '')[:80]
            print(f"   {i}. {insight_text}...")
        
        await memory.end_session()
        print("\n✅ Memory inspection complete")
        
    except Exception as e:
        print(f"\n⚠️  Error during inspection: {type(e).__name__}")
        print(f"   {str(e)[:150]}")

# Run memory inspection using await (not asyncio.run - Jupyter already has event loop)
try:
    await inspect_memory()
except Exception as e:
    print(f"❌ Inspection failed: {e}")

print("\n✅ Step 9 Complete: Memory validated")

STEP 9: Inspect Stored Memory
[Orchestrator] Starting session 08555b39-c4ae-489f-b280-273a39444c3f
[MemoryKeeper] Initializing session context for user: sarah_demo
  ✓ Loaded long-term insight profile (4412 chars)
     Preview: ## Overview
User ID: sarah_demo

Sarah is a long-term, growth-oriented investor building a retirement portfolio over roughly three decades. She is com...
     📋 Session 21b71d79...: The conversation focused on whether Roth conversions are appropriate for the user given their age, i...
     📋 Session ac193e61...: The conversation focused on designing an asset allocation for the user’s long‑term retirement invest...
     📋 Session d985466e...: The user, a 35-year-old software engineer earning $150k, shared that she has a 30-year horizon until...
  ✓ Loaded 3 recent session summaries

🔍 Searching: 'What is Sarah's risk tolerance?'
   ✅ Found: [Conversation] The user, Sarah, a 35-year-old software engineer earning $150k, expressed a moderate-to-high risk tolerance w

## Step 10: Cleanup and Summary
Close memory connections and clean up the database. Summarize key learnings about automatic memory management with Agent Framework.

In [14]:
print("=" * 70)
print("STEP 10: Cleanup and Summary")
print("=" * 70)

# Close memory connection
async def cleanup():
    try:
        await memory.close()
        print("\n✅ Memory connection closed")
    except Exception as e:
        print(f"\n⚠️  Error closing memory: {type(e).__name__}")

# Run cleanup using await (not asyncio.run - Jupyter already has event loop)
try:
    await cleanup()
except Exception as e:
    print(f"⚠️  Cleanup error: {e}")

# Clean up database (with retry for Windows file locks)
print(f"\n🧹 Cleaning up database: {DB_PATH}")
for attempt in range(5):
    try:
        if os.path.exists(DB_PATH):
            os.remove(DB_PATH)
            print(f"✅ Database deleted")
        break
    except PermissionError:
        if attempt < 4:
            time.sleep(0.5)
        else:
            print(f"⚠️  Could not delete database (in use)")

# Summary
print("\n" + "=" * 70)
print("📚 KEY LEARNINGS")
print("=" * 70)
print("""
✅ AUTOMATIC MEMORY MANAGEMENT WITH AGENT FRAMEWORK

1. ✓ ContextProvider Pattern
   - Pass memory as context_providers=[memory] to Agent
   - No manual add_turn() calls needed

2. ✓ Automatic Context Injection
   - before_run() loads context before each agent call
   - Agent uses previous conversations for personalization

3. ✓ Automatic Turn Capture
   - after_run() stores each response automatically
   - No explicit storage code required

4. ✓ Multi-Session Learning
   - Agent recalls details across conversations
   - Builds and updates user profiles
   - Extracts insights automatically

5. ✓ Production Ready
   - SQLite backend for local development
   - Vector embeddings for semantic search
   - Keyword-triggered auto-enrichment

HOW IT WORKED:
──────────────
• Session 1: Built Sarah's profile (age, income, risk tolerance)
• Session 2: Agent automatically recalled context from Session 1
• Session 3: Agent used knowledge from Sessions 1 & 2 for tax advice

All memory management was AUTOMATIC through context_providers!
""")
print("=" * 70)
print("\n✅ Notebook Complete! All steps finished successfully.")

STEP 10: Cleanup and Summary

✅ Memory connection closed

🧹 Cleaning up database: c:\LabFiles\agent-memory\demo_financial_advisor.db
✅ Database deleted

📚 KEY LEARNINGS

✅ AUTOMATIC MEMORY MANAGEMENT WITH AGENT FRAMEWORK

1. ✓ ContextProvider Pattern
   - Pass memory as context_providers=[memory] to Agent
   - No manual add_turn() calls needed

2. ✓ Automatic Context Injection
   - before_run() loads context before each agent call
   - Agent uses previous conversations for personalization

3. ✓ Automatic Turn Capture
   - after_run() stores each response automatically
   - No explicit storage code required

4. ✓ Multi-Session Learning
   - Agent recalls details across conversations
   - Builds and updates user profiles
   - Extracts insights automatically

5. ✓ Production Ready
   - SQLite backend for local development
   - Vector embeddings for semantic search
   - Keyword-triggered auto-enrichment

HOW IT WORKED:
──────────────
• Session 1: Built Sarah's profile (age, income, risk to